# ISIC 2019 Dataset Preparation and Verification

## Multi-Class Medical Image Classification

This notebook prepares the ISIC 2019 skin lesion dataset for all deep learning experiments in this project.

### Objectives
- Verify the raw dataset structure
- Inspect ground-truth labels
- Identify diagnostic classes
- Count available images
- Analyze class distribution
- Detect missing or invalid image references
- Create a reproducible stratified train/validation/test split
- Save the split for consistent use across all models

### Dataset
**ISIC 2019 Skin Lesion Classification Dataset**

The processed split generated in this notebook will be reused for:
- ResNet50
- DenseNet121
- Transfer Learning
- Data Augmentation experiments
- Custom CNN experiments
- Hybrid and Ensemble models

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

print("Libraries imported successfully.")

In [ ]:
# ---------------------------------------------------------
# Project Paths
# ---------------------------------------------------------

PROJECT_DIR = Path.cwd()

DATA_DIR = PROJECT_DIR / "dataset"
RAW_DATA_DIR = DATA_DIR / "ISIC_2019_raw"

IMAGE_DIR = IMAGE_DIR = (
    RAW_DATA_DIR
    / "ISIC_2019_Training_Input"
    / "ISIC_2019_Training_Input"
)
GROUND_TRUTH_FILE = RAW_DATA_DIR / "ISIC_2019_Training_GroundTruth.csv"
METADATA_FILE = RAW_DATA_DIR / "ISIC_2019_Training_Metadata.csv"

print("Project directory :", PROJECT_DIR)
print("Raw data directory:", RAW_DATA_DIR)
print("Image directory   :", IMAGE_DIR)

print("\nPath verification:")
print("Image folder exists :", IMAGE_DIR.exists())
print("Ground truth exists :", GROUND_TRUTH_FILE.exists())
print("Metadata exists     :", METADATA_FILE.exists())

In [ ]:
# ---------------------------------------------------------
# Load Ground-Truth Labels
# ---------------------------------------------------------

ground_truth_df = pd.read_csv(GROUND_TRUTH_FILE)

print("Ground-truth file loaded successfully.")
print("Dataset shape:", ground_truth_df.shape)

ground_truth_df.head()

In [ ]:
# ---------------------------------------------------------
# Inspect Dataset Columns
# ---------------------------------------------------------

print("Columns in the ground-truth dataset:\n")

for i, column in enumerate(ground_truth_df.columns, start=1):
    print(f"{i}. {column}")

In [ ]:
# ---------------------------------------------------------
# Basic Dataset Information
# ---------------------------------------------------------

print("Number of records :", len(ground_truth_df))
print("Number of columns :", len(ground_truth_df.columns))

print("\nMissing values per column:")
print(ground_truth_df.isnull().sum())

print("\nDuplicate rows:")
print(ground_truth_df.duplicated().sum())

In [ ]:
# ---------------------------------------------------------
# Verify One-Hot Encoded Labels
# ---------------------------------------------------------

label_columns = [
    col for col in ground_truth_df.columns
    if col != "image"
]

label_sum_per_image = ground_truth_df[label_columns].sum(axis=1)

print("Unique number of positive labels per image:")
print(label_sum_per_image.value_counts().sort_index())

In [ ]:
# ---------------------------------------------------------
# Class Name Mapping
# ---------------------------------------------------------

class_name_map = {
    "MEL": "Melanoma",
    "NV": "Melanocytic Nevus",
    "BCC": "Basal Cell Carcinoma",
    "AK": "Actinic Keratosis",
    "BKL": "Benign Keratosis",
    "DF": "Dermatofibroma",
    "VASC": "Vascular Lesion",
    "SCC": "Squamous Cell Carcinoma",
    "UNK": "Unknown"
}

print("Class mapping defined successfully.")

In [ ]:
# ---------------------------------------------------------
# Class Distribution
# ---------------------------------------------------------

class_counts = ground_truth_df[label_columns].sum().astype(int)

class_distribution_df = pd.DataFrame({
    "Class_Code": class_counts.index,
    "Class_Name": [
        class_name_map.get(code, code)
        for code in class_counts.index
    ],
    "Image_Count": class_counts.values
})

class_distribution_df["Percentage"] = (
    class_distribution_df["Image_Count"]
    / len(ground_truth_df)
    * 100
)

class_distribution_df = class_distribution_df.sort_values(
    by="Image_Count",
    ascending=False
).reset_index(drop=True)

class_distribution_df

In [ ]:
# ---------------------------------------------------------
# Print Class Distribution
# ---------------------------------------------------------

print("=" * 70)
print("ISIC 2019 CLASS DISTRIBUTION")
print("=" * 70)

for _, row in class_distribution_df.iterrows():
    print(
        f"{row['Class_Code']:5s} | "
        f"{row['Class_Name']:30s} | "
        f"{row['Image_Count']:6d} images | "
        f"{row['Percentage']:6.2f}%"
    )

In [ ]:
# ---------------------------------------------------------
# Visualize Class Distribution
# ---------------------------------------------------------

plt.figure(figsize=(12, 6))

plt.bar(
    class_distribution_df["Class_Name"],
    class_distribution_df["Image_Count"]
)

plt.title("ISIC 2019 Class Distribution")
plt.xlabel("Disease Class")
plt.ylabel("Number of Images")

plt.xticks(rotation=45, ha="right")

plt.tight_layout()
plt.show()

In [ ]:
# ---------------------------------------------------------
# Check Unknown Class
# ---------------------------------------------------------

if "UNK" in ground_truth_df.columns:
    unknown_count = int(ground_truth_df["UNK"].sum())

    print("Unknown-class images:", unknown_count)

    if unknown_count == 0:
        print("No images belong to the UNK class.")
    else:
        print(
            "UNK contains images and must be considered "
            "before finalizing the classification classes."
        )
else:
    print("No UNK column found.")

In [ ]:
# ---------------------------------------------------------
# Inspect Image Files
# ---------------------------------------------------------

from collections import Counter

valid_extensions = {".jpg", ".jpeg", ".png"}

image_files = [
    path for path in IMAGE_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in valid_extensions
]

extension_counts = Counter(
    path.suffix.lower() for path in image_files
)

print("Total image files found:", len(image_files))
print("\nFile extensions:")

for extension, count in extension_counts.items():
    print(f"{extension}: {count}")

In [ ]:
# ---------------------------------------------------------
# Compare Ground Truth with Image Files
# ---------------------------------------------------------

expected_image_ids = set(
    ground_truth_df["image"].astype(str)
)

actual_image_ids = set(
    path.stem for path in image_files
)

missing_images = sorted(
    expected_image_ids - actual_image_ids
)

unexpected_images = sorted(
    actual_image_ids - expected_image_ids
)

print("Images expected from CSV :", len(expected_image_ids))
print("Images physically found  :", len(actual_image_ids))

print("\nMissing images    :", len(missing_images))
print("Unexpected images :", len(unexpected_images))

In [ ]:
# ---------------------------------------------------------
# Check Duplicate Image IDs
# ---------------------------------------------------------

image_id_counts = Counter(
    path.stem for path in image_files
)

duplicate_image_ids = {
    image_id: count
    for image_id, count in image_id_counts.items()
    if count > 1
}

print("Duplicate image IDs:", len(duplicate_image_ids))

if duplicate_image_ids:
    print("\nExample duplicates:")
    
    for image_id, count in list(duplicate_image_ids.items())[:10]:
        print(image_id, "->", count)
else:
    print("No duplicate image IDs found.")

In [ ]:
# ---------------------------------------------------------
# Image Integrity Check
# ---------------------------------------------------------

corrupted_images = []

for index, image_path in enumerate(image_files, start=1):
    try:
        with Image.open(image_path) as img:
            img.verify()

    except Exception as error:
        corrupted_images.append(
            (image_path.name, str(error))
        )

    if index % 5000 == 0:
        print(
            f"Checked {index:,} / "
            f"{len(image_files):,} images"
        )

print("\n" + "=" * 60)
print("IMAGE INTEGRITY CHECK")
print("=" * 60)

print("Total checked    :", len(image_files))
print("Corrupted images :", len(corrupted_images))

In [ ]:
# ---------------------------------------------------------
# Inspect Raw Image Properties
# ---------------------------------------------------------

sample_files = image_files[:500]

image_sizes = []
image_modes = []

for image_path in sample_files:
    with Image.open(image_path) as img:
        image_sizes.append(img.size)
        image_modes.append(img.mode)

size_counts = Counter(image_sizes)
mode_counts = Counter(image_modes)

print("Images inspected:", len(sample_files))

print("\nImage modes:")
for mode, count in mode_counts.items():
    print(f"{mode}: {count}")

print("\nMost common image dimensions:")
for size, count in size_counts.most_common(10):
    print(f"{size}: {count}")

In [ ]:
# ---------------------------------------------------------
# Final Diagnostic Classes
# ---------------------------------------------------------

DISEASE_CLASSES = [
    "MEL",
    "NV",
    "BCC",
    "AK",
    "BKL",
    "DF",
    "VASC",
    "SCC"
]

class_name_map = {
    "MEL": "Melanoma",
    "NV": "Melanocytic Nevus",
    "BCC": "Basal Cell Carcinoma",
    "AK": "Actinic Keratosis",
    "BKL": "Benign Keratosis",
    "DF": "Dermatofibroma",
    "VASC": "Vascular Lesion",
    "SCC": "Squamous Cell Carcinoma"
}

print("Number of diagnostic classes:", len(DISEASE_CLASSES))

for code in DISEASE_CLASSES:
    print(f"{code:5s} -> {class_name_map[code]}")

In [ ]:
# ---------------------------------------------------------
# Convert One-Hot Labels to Single Class Labels
# ---------------------------------------------------------

master_df = ground_truth_df.copy()

master_df["class_code"] = master_df[DISEASE_CLASSES].idxmax(axis=1)

master_df["class_name"] = master_df["class_code"].map(
    class_name_map
)

master_df[[
    "image",
    "class_code",
    "class_name"
]].head(10)

In [ ]:
# ---------------------------------------------------------
# Verify Label Conversion
# ---------------------------------------------------------

positive_labels_per_image = master_df[
    DISEASE_CLASSES
].sum(axis=1)

invalid_label_rows = master_df[
    positive_labels_per_image != 1
]

print("Total records          :", len(master_df))
print("Valid single-label rows:", (positive_labels_per_image == 1).sum())
print("Invalid label rows     :", len(invalid_label_rows))

In [ ]:
# ---------------------------------------------------------
# Create Numerical Class Mapping
# ---------------------------------------------------------

class_to_index = {
    class_code: index
    for index, class_code in enumerate(DISEASE_CLASSES)
}

index_to_class = {
    index: class_code
    for class_code, index in class_to_index.items()
}

master_df["class_index"] = master_df[
    "class_code"
].map(class_to_index)

print("Class-to-index mapping:\n")

for class_code, index in class_to_index.items():
    print(
        f"{index} -> {class_code:5s} -> "
        f"{class_name_map[class_code]}"
    )

In [ ]:
# ---------------------------------------------------------
# Attach Physical Image Paths
# ---------------------------------------------------------

image_path_map = {
    path.stem: str(path.resolve())
    for path in image_files
}

master_df["image_path"] = master_df["image"].map(
    image_path_map
)

print(
    "Missing image paths:",
    master_df["image_path"].isna().sum()
)

master_df[[
    "image",
    "image_path",
    "class_code",
    "class_index"
]].head()

In [ ]:
# ---------------------------------------------------------
# Load ISIC Metadata
# ---------------------------------------------------------

metadata_df = pd.read_csv(METADATA_FILE)

print("Metadata shape:", metadata_df.shape)

print("\nMetadata columns:")
for column in metadata_df.columns:
    print(column)

metadata_df.head()

In [ ]:
# ---------------------------------------------------------
# Merge Labels and Metadata
# ---------------------------------------------------------

metadata_columns = ["image"]

if "lesion_id" in metadata_df.columns:
    metadata_columns.append("lesion_id")

if "age_approx" in metadata_df.columns:
    metadata_columns.append("age_approx")

if "sex" in metadata_df.columns:
    metadata_columns.append("sex")

if "anatom_site_general" in metadata_df.columns:
    metadata_columns.append("anatom_site_general")


master_df = master_df.merge(
    metadata_df[metadata_columns],
    on="image",
    how="left"
)

print("Master dataframe shape:", master_df.shape)

master_df.head()

In [ ]:
# ---------------------------------------------------------
# Inspect Lesion Grouping
# ---------------------------------------------------------

if "lesion_id" in master_df.columns:

    print(
        "Missing lesion IDs:",
        master_df["lesion_id"].isna().sum()
    )

    print(
        "Unique lesion IDs:",
        master_df["lesion_id"].nunique()
    )

    lesion_image_counts = master_df[
        "lesion_id"
    ].value_counts()

    print(
        "\nLesions represented by more than one image:",
        (lesion_image_counts > 1).sum()
    )

    print("\nImages per lesion:")
    print(lesion_image_counts.describe())

else:
    print("No lesion_id column is available.")

In [ ]:
# ---------------------------------------------------------
# Final Master Dataset
# ---------------------------------------------------------

final_columns = [
    "image",
    "image_path",
    "class_code",
    "class_name",
    "class_index"
]

for optional_column in [
    "lesion_id",
    "age_approx",
    "sex",
    "anatom_site_general"
]:
    if optional_column in master_df.columns:
        final_columns.append(optional_column)


master_df = master_df[final_columns].copy()

print("Final master dataset shape:", master_df.shape)

master_df.head()

In [ ]:
# ---------------------------------------------------------
# Master Dataset Summary
# ---------------------------------------------------------

print("=" * 70)
print("MASTER DATASET SUMMARY")
print("=" * 70)

print(f"Total images  : {len(master_df):,}")
print(f"Total classes : {master_df['class_code'].nunique()}")

print("\nClass distribution:")

summary = (
    master_df
    .groupby(["class_index", "class_code", "class_name"])
    .size()
    .reset_index(name="images")
)

print(summary.to_string(index=False))

In [ ]:
# ---------------------------------------------------------
# Save Master Dataset Table
# ---------------------------------------------------------

MASTER_FILE = DATA_DIR / "ISIC_2019_master.csv"

master_df.to_csv(
    MASTER_FILE,
    index=False
)

print("Master dataset saved successfully.")
print("Location:", MASTER_FILE)

In [ ]:
# ---------------------------------------------------------
# Import Dataset Splitting Utility
# ---------------------------------------------------------

from sklearn.model_selection import train_test_split

RANDOM_SEED = 42

print("Random seed:", RANDOM_SEED)

In [ ]:
# ---------------------------------------------------------
# Create Group IDs for Leakage-Safe Splitting
# ---------------------------------------------------------

split_df = master_df.copy()

if "lesion_id" in split_df.columns:

    split_df["split_group"] = split_df["lesion_id"].astype("string")

    missing_lesion_mask = split_df["split_group"].isna()

    split_df.loc[
        missing_lesion_mask,
        "split_group"
    ] = (
        "IMAGE_" +
        split_df.loc[missing_lesion_mask, "image"]
    )

else:

    split_df["split_group"] = (
        "IMAGE_" + split_df["image"]
    )

print("Total images :", len(split_df))
print("Total groups :", split_df["split_group"].nunique())

In [ ]:
# ---------------------------------------------------------
# Verify Group/Class Consistency
# ---------------------------------------------------------

group_class_counts = (
    split_df
    .groupby("split_group")["class_code"]
    .nunique()
)

conflicting_groups = group_class_counts[
    group_class_counts > 1
]

print(
    "Groups containing multiple diagnostic classes:",
    len(conflicting_groups)
)

In [ ]:
# ---------------------------------------------------------
# Build Group-Level Dataset
# ---------------------------------------------------------

group_df = (
    split_df
    .groupby("split_group")
    .agg(
        class_code=("class_code", "first"),
        image_count=("image", "size")
    )
    .reset_index()
)

print("Number of groups:", len(group_df))

print("\nGroup-level class distribution:")
print(
    group_df["class_code"]
    .value_counts()
)

In [ ]:
# ---------------------------------------------------------
# Stage 1: Train vs Temporary Split
# ---------------------------------------------------------

train_groups, temp_groups = train_test_split(
    group_df,
    test_size=0.30,
    random_state=RANDOM_SEED,
    stratify=group_df["class_code"]
)

print("Train groups :", len(train_groups))
print("Temp groups  :", len(temp_groups))

In [ ]:
# ---------------------------------------------------------
# Stage 2: Validation vs Test Split
# ---------------------------------------------------------

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    random_state=RANDOM_SEED,
    stratify=temp_groups["class_code"]
)

print("Train groups      :", len(train_groups))
print("Validation groups :", len(val_groups))
print("Test groups       :", len(test_groups))

In [ ]:
# ---------------------------------------------------------
# Assign Split Labels
# ---------------------------------------------------------

train_group_ids = set(train_groups["split_group"])
val_group_ids = set(val_groups["split_group"])
test_group_ids = set(test_groups["split_group"])


def assign_split(group_id):

    if group_id in train_group_ids:
        return "train"

    elif group_id in val_group_ids:
        return "val"

    elif group_id in test_group_ids:
        return "test"

    return "unknown"


split_df["split"] = split_df[
    "split_group"
].apply(assign_split)

print(split_df["split"].value_counts())

In [ ]:
# ---------------------------------------------------------
# Verify No Group Leakage
# ---------------------------------------------------------

train_set = set(
    split_df.loc[
        split_df["split"] == "train",
        "split_group"
    ]
)

val_set = set(
    split_df.loc[
        split_df["split"] == "val",
        "split_group"
    ]
)

test_set = set(
    split_df.loc[
        split_df["split"] == "test",
        "split_group"
    ]
)

print(
    "Train ↔ Validation overlap:",
    len(train_set & val_set)
)

print(
    "Train ↔ Test overlap:",
    len(train_set & test_set)
)

print(
    "Validation ↔ Test overlap:",
    len(val_set & test_set)
)

In [ ]:
# ---------------------------------------------------------
# Verify Complete Split Assignment
# ---------------------------------------------------------

print("Total images:", len(split_df))

print(
    "Assigned images:",
    split_df["split"]
    .isin(["train", "val", "test"])
    .sum()
)

print(
    "Unknown assignments:",
    (split_df["split"] == "unknown").sum()
)

In [ ]:
# ---------------------------------------------------------
# Class Distribution Across Splits
# ---------------------------------------------------------

split_distribution = pd.crosstab(
    split_df["class_name"],
    split_df["split"]
)

split_distribution["Total"] = (
    split_distribution.sum(axis=1)
)

split_distribution

In [ ]:
# ---------------------------------------------------------
# Split Statistics
# ---------------------------------------------------------

split_counts = (
    split_df["split"]
    .value_counts()
)

print("=" * 60)
print("FINAL DATASET SPLIT")
print("=" * 60)

for split_name in ["train", "val", "test"]:

    count = split_counts[split_name]

    percentage = (
        count / len(split_df)
    ) * 100

    print(
        f"{split_name.upper():10s}: "
        f"{count:6,d} images "
        f"({percentage:5.2f}%)"
    )

In [ ]:
# ---------------------------------------------------------
# Create Split Directory
# ---------------------------------------------------------

SPLIT_DIR = DATA_DIR / "splits"

SPLIT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

print("Split directory:")
print(SPLIT_DIR)

In [ ]:
# ---------------------------------------------------------
# Save Permanent Dataset Splits
# ---------------------------------------------------------

train_df = split_df[
    split_df["split"] == "train"
].copy()

val_df = split_df[
    split_df["split"] == "val"
].copy()

test_df = split_df[
    split_df["split"] == "test"
].copy()


train_df.to_csv(
    SPLIT_DIR / "train.csv",
    index=False
)

val_df.to_csv(
    SPLIT_DIR / "val.csv",
    index=False
)

test_df.to_csv(
    SPLIT_DIR / "test.csv",
    index=False
)

split_df.to_csv(
    SPLIT_DIR / "all_splits.csv",
    index=False
)

print("Dataset splits saved successfully.")

print("\nFiles created:")
print(SPLIT_DIR / "train.csv")
print(SPLIT_DIR / "val.csv")
print(SPLIT_DIR / "test.csv")
print(SPLIT_DIR / "all_splits.csv")

In [ ]:
# ---------------------------------------------------------
# Save Split Configuration
# ---------------------------------------------------------

import json

split_config = {
    "dataset": "ISIC 2019",
    "number_of_classes": 8,
    "classes": DISEASE_CLASSES,
    "class_to_index": class_to_index,
    "random_seed": RANDOM_SEED,
    "target_train_ratio": 0.70,
    "target_validation_ratio": 0.15,
    "target_test_ratio": 0.15,
    "grouped_by_lesion": "lesion_id" in split_df.columns,
    "total_images": len(split_df),
    "train_images": len(train_df),
    "validation_images": len(val_df),
    "test_images": len(test_df)
}

CONFIG_FILE = SPLIT_DIR / "split_config.json"

with open(CONFIG_FILE, "w") as file:
    json.dump(
        split_config,
        file,
        indent=4
    )

print("Split configuration saved:")
print(CONFIG_FILE)

In [ ]:
# ---------------------------------------------------------
# Reload Saved Dataset Splits
# ---------------------------------------------------------

train_check = pd.read_csv(SPLIT_DIR / "train.csv")
val_check = pd.read_csv(SPLIT_DIR / "val.csv")
test_check = pd.read_csv(SPLIT_DIR / "test.csv")

print("Saved split files loaded successfully.\n")

print(f"Training images   : {len(train_check):,}")
print(f"Validation images : {len(val_check):,}")
print(f"Test images       : {len(test_check):,}")

print(f"\nTotal images      : {len(train_check) + len(val_check) + len(test_check):,}")

In [ ]:
# ---------------------------------------------------------
# Verify Classes in Every Split
# ---------------------------------------------------------

for split_name, dataframe in {
    "Train": train_check,
    "Validation": val_check,
    "Test": test_check
}.items():

    classes = sorted(dataframe["class_code"].unique())

    print(f"{split_name}:")
    print("Classes:", classes)
    print("Number of classes:", len(classes))
    print()

In [ ]:
# ---------------------------------------------------------
# Final Leakage Verification
# ---------------------------------------------------------

train_groups_check = set(train_check["split_group"])
val_groups_check = set(val_check["split_group"])
test_groups_check = set(test_check["split_group"])

print("Train ↔ Validation overlap:",
      len(train_groups_check & val_groups_check))

print("Train ↔ Test overlap:",
      len(train_groups_check & test_groups_check))

print("Validation ↔ Test overlap:",
      len(val_groups_check & test_groups_check))

In [ ]:
# ---------------------------------------------------------
# Final Class Distribution Table
# ---------------------------------------------------------

final_distribution = pd.DataFrame({
    "Train": train_check["class_name"].value_counts(),
    "Validation": val_check["class_name"].value_counts(),
    "Test": test_check["class_name"].value_counts()
}).fillna(0).astype(int)

final_distribution["Total"] = final_distribution.sum(axis=1)

final_distribution

In [ ]:
# ---------------------------------------------------------
# Display Sample Images from Each Class
# ---------------------------------------------------------

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

axes = axes.flatten()

for index, class_code in enumerate(DISEASE_CLASSES):

    sample_row = train_check[
        train_check["class_code"] == class_code
    ].sample(
        n=1,
        random_state=RANDOM_SEED
    ).iloc[0]

    image = Image.open(sample_row["image_path"])

    axes[index].imshow(image)
    axes[index].set_title(
        f"{class_code}\n{sample_row['class_name']}"
    )
    axes[index].axis("off")

plt.suptitle(
    "Sample Images from the 8 ISIC 2019 Diagnostic Classes",
    fontsize=16
)

plt.tight_layout()
plt.show()

## Dataset Preparation Completed

The ISIC 2019 dataset has been successfully prepared for the deep learning experiments.

### Final Dataset Configuration

- **Total images:** 25,331
- **Number of diagnostic classes:** 8
- **Image format:** RGB
- **Training/Validation/Test splitting:** lesion-aware and stratified
- **Random seed:** 42
- **Data leakage prevention:** images belonging to the same lesion are kept within the same split
- **Split files:** permanently saved and reused across all experiments

All subsequent models will use the same train, validation, and test sets to ensure a fair comparison.